# Week 10 Walkthrough — Fetching Live Data

**Topic:** HTTP requests, working with APIs, parsing JSON, error handling

Until now every byte of data was one you typed. This week it comes from somewhere else. We'll call a real API, read the JSON it returns, and — crucially — write the code so it still works when the network doesn't.

Run the cells in order. Each step adds one idea to the program, and the last
section pulls the whole thing together. Change things and re-run — that is the
whole point of a notebook.

## Step 1 — A request in three lines


`requests.get()` fetches, `.status_code` reports how it went, `.json()` turns
the body into Python objects.

Everything here uses **JSONPlaceholder**, a free practice API with no key and no
signup.

In [ ]:
import requests

BASE = "https://jsonplaceholder.typicode.com"

try:
    response = requests.get(f"{BASE}/posts/1", timeout=5)
    print("status:", response.status_code, "| ok:", response.ok)
    online = response.ok
except requests.exceptions.RequestException as e:
    print("No network right now:", e.__class__.__name__)
    online = False

## Step 2 — Sample data, so the notebook always runs


A habit worth forming: keep a small local copy of what the API returns. Your
notebook still runs on a train, and you can develop without hammering someone
else's server.

In [ ]:
SAMPLE_POSTS = [
    {"userId": 1, "id": 1, "title": "sunt aut facere", "body": "quia et suscipit"},
    {"userId": 1, "id": 2, "title": "qui est esse", "body": "est rerum tempore"},
    {"userId": 2, "id": 11, "title": "et ea vero quia", "body": "delectus reiciendis"},
    {"userId": 2, "id": 12, "title": "in quibusdam tempore", "body": "quo deleniti"},
    {"userId": 3, "id": 21, "title": "asperiores ea ipsam", "body": "repellat molestiae"},
]

def fetch_posts():
    """Return posts from the API, or the local sample if that fails."""
    if not online:
        print("(using local sample)")
        return SAMPLE_POSTS
    try:
        r = requests.get(f"{BASE}/posts", timeout=5)
        r.raise_for_status()
        return r.json()
    except requests.exceptions.RequestException:
        print("(request failed - using local sample)")
        return SAMPLE_POSTS


posts = fetch_posts()
print(len(posts), "posts")

## Step 3 — JSON is dictionaries and lists


Nothing new to learn. A JSON object is a `dict`, a JSON array is a `list`, and
everything from Week 6 and 7 applies.

In [ ]:
first = posts[0]

print(type(posts), type(first))
print(first.keys())
print(first["title"])

## Step 4 — Look before you guess


When you don't know the shape of a response, print it indented. This is the
fastest way to understand an unfamiliar API.

In [ ]:
import json

print(json.dumps(posts[0], indent=2))

## Step 5 — Query parameters


Let `requests` build the URL from a dictionary. Never glue query strings
together by hand — encoding will bite you.

In [ ]:
def fetch_posts_by(user_id):
    if not online:
        return [p for p in SAMPLE_POSTS if p["userId"] == user_id]
    try:
        r = requests.get(f"{BASE}/posts", params={"userId": user_id}, timeout=5)
        r.raise_for_status()
        print("requested:", r.url)
        return r.json()
    except requests.exceptions.RequestException:
        return [p for p in SAMPLE_POSTS if p["userId"] == user_id]


mine = fetch_posts_by(1)
print(len(mine), "posts by user 1")

## Step 6 — Counting, exactly as in Week 7


The data came from the internet. The analysis is the same dictionary pattern you
already know.

In [ ]:
by_author = {}
for post in posts:
    author = post["userId"]
    by_author[author] = by_author.get(author, 0) + 1

for author, n in sorted(by_author.items(), key=lambda p: p[1], reverse=True):
    print(f"user {author}: {n} posts")

## Step 7 — Be a good citizen


Pause between calls in a loop, and save what you fetch so re-running your
analysis doesn't re-hit the server.

In [ ]:
import csv
import time

rows = [{"id": p["id"], "userId": p["userId"], "title": p["title"]} for p in posts[:5]]

with open("posts.csv", "w", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=["id", "userId", "title"])
    writer.writeheader()
    writer.writerows(rows)

time.sleep(0.2)     # in a real loop: pause between requests
print(f"saved {len(rows)} rows to posts.csv")

---

## The finished program

Everything above, in one place. This is the version worth keeping.


Fetch with a fallback, summarise, save. Run it with the wifi off — it still
works, and it tells you why.

In [ ]:
# Week 10 - Fetch, summarise, store

import csv
import json
import requests

BASE = "https://jsonplaceholder.typicode.com"


def get_json(path, params=None, fallback=None):
    """GET and parse JSON, falling back to local data on any failure."""
    try:
        r = requests.get(f"{BASE}{path}", params=params, timeout=5)
        r.raise_for_status()
        return r.json(), "live"
    except requests.exceptions.Timeout:
        reason = "timed out"
    except requests.exceptions.ConnectionError:
        reason = "could not connect"
    except requests.exceptions.HTTPError as e:
        reason = f"server said {e.response.status_code}"
    except ValueError:
        reason = "response was not JSON"
    return fallback, reason


posts, source = get_json("/posts", fallback=SAMPLE_POSTS)

print(f"SOURCE: {source}")
print("=" * 46)

counts = {}
for post in posts:
    counts[post["userId"]] = counts.get(post["userId"], 0) + 1

print(f"{len(posts)} posts from {len(counts)} authors\n")
for author, n in sorted(counts.items(), key=lambda p: p[1], reverse=True)[:5]:
    bar = "#" * n
    print(f"user {author:2}  {n:3}  {bar}")

longest = max(posts, key=lambda p: len(p["title"]))
print(f"\nLongest title ({len(longest['title'])} chars): {longest['title']}")

with open("posts_summary.json", "w") as file:
    json.dump({"source": source, "counts": counts}, file, indent=2)
print("\nsummary written to posts_summary.json")

---

## Try it yourself

Use the empty cells below. There is no grade attached — this is where the
learning actually happens.

**1.** Turn your wifi off and re-run the finished cell. It should report the fallback and keep going.

**2.** Fetch `/users` instead and print each user's name and city — remember the city is nested inside `address`.

**3.** Add a `get_json('/posts/999999')` call. Which except branch catches it?

**4.** Write `comments_on(post_id)` using `params={'postId': post_id}`.

In [ ]:
# Try it yourself 1

In [ ]:
# Try it yourself 2

In [ ]:
# Try it yourself 3